In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv
import os

# Load .env
load_dotenv(find_dotenv())

DB_URL = os.getenv("LOCAL_DATABASE_URL")

if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace(
        "postgresql://",
        "postgresql+psycopg2://",
        1
    )

engine = create_engine(DB_URL)

print("Database connection berhasil")
print(f"Target: {DB_URL.split('@')[1] if DB_URL else 'NONE'}")

Database connection berhasil
Target: localhost:5432/retail_analytics


In [2]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS silver"))

print("Schema silver siap")

Schema silver siap


In [3]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'bronze'
ORDER BY table_name;
"""

bronze_tables = pd.read_sql(query, engine)

print("Tabel di Bronze:")
display(bronze_tables)

Tabel di Bronze:


,table_name
0,campaign_spend
1,city_reference
2,customer_addresses
3,customer_profiles
4,customers
5,inventory_snapshots
6,order_items
7,order_promotions
8,orders
9,payment_events


In [4]:
df_bronze = pd.read_sql(
    """
    SELECT raw, timestamp
    FROM bronze.customers
    """,
    engine
)

print(f"Jumlah data Bronze: {len(df_bronze)}")

display(df_bronze.head())

Jumlah data Bronze: 2500


,raw,timestamp
0,"{'city_id': 'JKT', 'customer_id': 'CUST-00001'...",2026-09-22 20:52:04.880993+00:00
1,"{'city_id': 'MDN', 'customer_id': 'CUST-00002'...",2026-09-22 20:52:04.880993+00:00
2,"{'city_id': 'BDG', 'customer_id': 'CUST-00003'...",2026-09-22 20:52:04.880993+00:00
3,"{'city_id': 'BDG', 'customer_id': 'CUST-00004'...",2026-09-22 20:52:04.880993+00:00
4,"{'city_id': 'JKT', 'customer_id': 'CUST-00005'...",2026-09-22 20:52:04.880993+00:00


In [5]:
df_silver = pd.json_normalize(df_bronze["raw"])

print("Kolom hasil parsing JSON:")
print(df_silver.columns.tolist())

display(df_silver.head())

Kolom hasil parsing JSON:
['city_id', 'customer_id', 'source_row_id', 'created_at_utc', 'customer_segment']


,city_id,customer_id,source_row_id,created_at_utc,customer_segment
0,JKT,CUST-00001,CUSTOMER-ROW-000001,2025-07-29T00:00:00Z,consumer
1,MDN,CUST-00002,CUSTOMER-ROW-000002,2026-02-01T00:00:00Z,consumer
2,BDG,CUST-00003,CUSTOMER-ROW-000003,2026-04-11T00:00:00Z,enterprise
3,BDG,CUST-00004,CUSTOMER-ROW-000004,2025-07-10T00:00:00Z,enterprise
4,JKT,CUST-00005,CUSTOMER-ROW-000005,2025-08-23T00:00:00Z,small_business


In [6]:
df_silver["ingested_at_utc"] = pd.to_datetime(
    df_bronze["timestamp"],
    utc=True
)

display(df_silver.head())

,city_id,customer_id,source_row_id,created_at_utc,customer_segment,ingested_at_utc
0,JKT,CUST-00001,CUSTOMER-ROW-000001,2025-07-29T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
1,MDN,CUST-00002,CUSTOMER-ROW-000002,2026-02-01T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
2,BDG,CUST-00003,CUSTOMER-ROW-000003,2026-04-11T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
3,BDG,CUST-00004,CUSTOMER-ROW-000004,2025-07-10T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
4,JKT,CUST-00005,CUSTOMER-ROW-000005,2025-08-23T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00


In [7]:
df_silver.to_sql(
    "customers",
    engine,
    schema="silver",
    if_exists="replace",
    index=False
)

print("silver.customers berhasil dibuat")
print(f"Jumlah rows: {len(df_silver)}")

silver.customers berhasil dibuat
Jumlah rows: 2500


In [8]:
df_check = pd.read_sql(
    """
    SELECT *
    FROM silver.customers
    LIMIT 10
    """,
    engine
)

display(df_check)

,city_id,customer_id,source_row_id,created_at_utc,customer_segment,ingested_at_utc
0,JKT,CUST-00001,CUSTOMER-ROW-000001,2025-07-29T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
1,MDN,CUST-00002,CUSTOMER-ROW-000002,2026-02-01T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
2,BDG,CUST-00003,CUSTOMER-ROW-000003,2026-04-11T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
3,BDG,CUST-00004,CUSTOMER-ROW-000004,2025-07-10T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
4,JKT,CUST-00005,CUSTOMER-ROW-000005,2025-08-23T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00
5,MDN,CUST-00006,CUSTOMER-ROW-000006,2026-06-06T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
6,JKT,CUST-00007,CUSTOMER-ROW-000007,2026-02-22T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
7,BPN,CUST-00008,CUSTOMER-ROW-000008,2025-09-07T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
8,DPS,CUST-00009,CUSTOMER-ROW-000009,2026-03-01T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00
9,MKS,CUST-00010,CUSTOMER-ROW-000010,2025-06-29T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00


In [9]:
print("Bronze :", len(df_bronze))
print("Silver :", len(df_silver))

Bronze : 2500
Silver : 2500


In [11]:
print("Tabel: silver.customers")
print("Kolom:")

for col in df_silver.columns:
    print("-", col)

Tabel: silver.customers
Kolom:
- city_id
- customer_id
- source_row_id
- created_at_utc
- customer_segment
- ingested_at_utc


In [16]:
def bronze_to_silver(table_name):
    print(f"\nProcessing: bronze.{table_name}")

    # Cek struktur kolom Bronze
    columns = pd.read_sql(
        f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'bronze'
          AND table_name = '{table_name}'
        ORDER BY ordinal_position
        """,
        engine
    )["column_name"].tolist()

    # Jika ada kolom raw → berarti source JSON
    if "raw" in columns:

        df_bronze = pd.read_sql(
            f"""
            SELECT raw, timestamp
            FROM bronze."{table_name}"
            """,
            engine
        )

        # JSON → kolom
        df_silver = pd.json_normalize(df_bronze["raw"])

        # Tambahkan timestamp ingestion
        df_silver["ingested_at_utc"] = pd.to_datetime(
            df_bronze["timestamp"],
            utc=True
        )

    # Jika tidak ada raw → berarti source sudah berupa kolom
    else:

        df_silver = pd.read_sql(
            f"""
            SELECT *
            FROM bronze."{table_name}"
            """,
            engine
        )

    # Simpan ke Silver
    df_silver.to_sql(
        table_name,
        engine,
        schema="silver",
        if_exists="replace",
        index=False
    )

    print(f"✓ silver.{table_name} → {len(df_silver)} rows")

    return df_silver

In [13]:
df_products = bronze_to_silver("products")


Processing: bronze.products
✓ silver.products → 80 rows


In [14]:
bronze_tables = pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
    """,
    engine
)

for table in bronze_tables["table_name"]:
    bronze_to_silver(table)


Processing: bronze.campaign_spend


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "raw" does not exist
LINE 2:         SELECT raw, timestamp
                       ^

[SQL: 
        SELECT raw, timestamp
        FROM bronze."campaign_spend"
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [15]:
pd.read_sql(
    """
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze'
      AND table_name = 'campaign_spend'
    ORDER BY ordinal_position
    """,
    engine
)

,column_name,data_type
0,spend_date,text
1,campaign_id,text
2,channel,text
3,spend_amount,double precision
4,timestamp,timestamp with time zone


In [17]:
df_campaign_spend = bronze_to_silver("campaign_spend")


Processing: bronze.campaign_spend
✓ silver.campaign_spend → 855 rows


In [18]:
for table in bronze_tables["table_name"]:
    try:
        bronze_to_silver(table)
    except Exception as e:
        print(f"❌ Gagal {table}: {e}")


Processing: bronze.campaign_spend
✓ silver.campaign_spend → 855 rows

Processing: bronze.city_reference
✓ silver.city_reference → 8 rows

Processing: bronze.customer_addresses
✓ silver.customer_addresses → 2500 rows

Processing: bronze.customer_profiles
✓ silver.customer_profiles → 2777 rows

Processing: bronze.customers
✓ silver.customers → 2500 rows

Processing: bronze.inventory_snapshots
✓ silver.inventory_snapshots → 35040 rows

Processing: bronze.order_items
✓ silver.order_items → 24960 rows

Processing: bronze.order_promotions
✓ silver.order_promotions → 15729 rows

Processing: bronze.orders
✓ silver.orders → 10001 rows

Processing: bronze.payment_events
✓ silver.payment_events → 22582 rows

Processing: bronze.product_categories
✓ silver.product_categories → 86 rows

Processing: bronze.products
✓ silver.products → 80 rows

Processing: bronze.promotions
✓ silver.promotions → 12 rows

Processing: bronze.refund_events
✓ silver.refund_events → 4121 rows

Processing: bronze.return_ev

In [2]:
import pandas as pd

In [2]:
schema_check = pd.read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema IN ('bronze', 'silver')
ORDER BY
    table_schema,
    table_name,
    ordinal_position
""", engine)

print("Total columns:", len(schema_check))
display(schema_check)

Total columns: 177


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,bronze,campaign_spend,1,spend_date,text,YES
1,bronze,campaign_spend,2,campaign_id,text,YES
2,bronze,campaign_spend,3,channel,text,YES
3,bronze,campaign_spend,4,spend_amount,double precision,YES
4,bronze,campaign_spend,5,timestamp,timestamp with time zone,YES
...,...,...,...,...,...,...
172,silver,web_events,5,payload.channel,text,YES
173,silver,web_events,6,payload.order_id,text,YES
174,silver,web_events,7,payload.session_id,text,YES
175,silver,web_events,8,payload.campaign_id,text,YES
